# Validation of the Deflection Angle

1. **Literature (Λ = 0).** Hasanuddin (2012) gives the deflection angle in Kerr spacetime for $a = 0.52$ and
   $a = 0.9939$, $b = 5$–$40$, with source and observer **at infinity**. The simulation starts at $r_0 = 500$ and
   ends at $r \approx 505$, so it is compared with the exact asymptotic value and with the exact finite-distance value.
2. **Exact benchmark (Λ ≠ 0).** The simulated deflection angle is compared with the exact value obtained by quadrature
   of the first integrals of the null geodesic, for prograde and retrograde spins and both signs of $\Lambda$.

See `docs/METHODOLOGY.md` (Section 9) for details.

In [1]:
import numpy as np

from kerr_ds import analytic, run_spin_sweep

M, E, r0, theta0, phi0 = 1.0, 1.0, 500.0, np.pi / 2, 0.0

## 1. Comparison with Hasanuddin (2012), Λ = 0

Columns (degrees): the value of Hasanuddin (2012, Tables 3–4); the exact asymptotic value
(`analytic.asymptotic_kerr_deflection_deg`); the simulated deflection angle $\delta$ at finite distance and its
exact counterpart; the coordinate value $\Delta\phi - \pi$ plus the flat-space offset
$\arcsin(b/r_S) + \arcsin(b/r_O)$; and the legacy value $\lvert\Delta\phi - \pi\rvert$ of the thesis tables.

In [2]:
spins = [0.52, 0.9939]
impacts = [5.0, 10.0, 15.0, 20.0, 40.0]
hasanuddin_rad = {0.52: [1.847027, 0.532207, 0.315107, 0.225507, 0.103067],
                  0.9939: [1.315827, 0.494747, 0.303387, 0.219707, 0.101867]}

runs = {}
for b in impacts:
    res = run_spin_sweep(M, spins, [0.0], r0, theta0, phi0, E, b * E, verbose=False)
    for a in spins:
        runs[a, b] = res[a, 0.0]

print(f"{'a':>6} {'b':>4} | {'Hasanuddin':>10} | {'exact asym.':>11} | {'delta sim':>10} | {'delta exact':>11} | "
      f"{'coord+offset':>12} | {'legacy':>8}")
print("-" * 90)
for a in spins:
    for i, b in enumerate(impacts):
        res = runs[a, b]
        r_obs = res.r[-1]
        offset = np.degrees(np.arcsin(b / r0) + np.arcsin(b / r_obs))
        print(f"{a:>6} {b:>4g} | {np.degrees(hasanuddin_rad[a][i]):>10.3f} | "
              f"{analytic.asymptotic_kerr_deflection_deg(M, a, b):>11.4f} | {res.deflection_deg:>10.4f} | "
              f"{analytic.deflection_exact_deg(M, a, 0.0, b, r0, r_obs):>11.4f} | "
              f"{res.deflection_coord_deg + offset:>12.4f} | {res.deflection_legacy_deg:>8.3f}")

     a    b | Hasanuddin | exact asym. |  delta sim | delta exact | coord+offset |   legacy
------------------------------------------------------------------------------------------
  0.52    5 |    105.827 |    105.9220 |   105.9200 |    105.9200 |     105.9218 |  104.782
  0.52   10 |     30.493 |     30.7755 |    30.7712 |     30.7712 |      30.7752 |   28.495
  0.52   15 |     18.054 |     18.3377 |    18.3311 |     18.3311 |      18.3375 |   14.916
  0.52   20 |     12.921 |     13.0814 |    13.0726 |     13.0726 |      13.0812 |    8.519
  0.52   40 |      5.905 |      6.1036 |     6.0856 |      6.0856 |       6.1034 |    3.028
0.9939    5 |     75.391 |     75.4716 |    75.4697 |     75.4697 |      75.4711 |   74.331
0.9939   10 |     28.347 |     28.4918 |    28.4877 |     28.4877 |      28.4914 |   26.211
0.9939   15 |     17.383 |     17.5678 |    17.5614 |     17.5614 |      17.5674 |   14.146
0.9939   20 |     12.588 |     12.6996 |    12.6910 |     12.6910 |      12.6992 

- The simulated $\delta$ equals the exact finite-distance value. It is slightly smaller than the asymptotic value
  because the bending accumulated beyond $r = 500$ is not included (about $2Mb/r_0^2$ rad, i.e. 0.002° for $b = 5$
  and 0.018° for $b = 40$).
- $\Delta\phi - \pi$ plus the flat-space offset reproduces the asymptotic value. The legacy value differs from the
  literature by this offset (1.1°–9.1°), which is a matter of definition, not a numerical error; for $b = 40$ the
  coordinate value $\Delta\phi - \pi$ is negative, and the legacy value is its absolute value.
- The values of Hasanuddin (2012) are 0.08°–0.28° below the exact asymptotic values (confirmed by the weak-field
  series, e.g. $a = 0.52$, $b = 40$: $\approx 6.10°$).

## 2. Exact benchmark for Λ ≠ 0 (b = 7)

In [3]:
b = 7.0
res = run_spin_sweep(M, [0.0, 0.99, -0.99], [-5e-6, 0.0, 5e-6], r0, theta0, phi0, E, b * E, verbose=False)

print(f"{'a':>6} {'Lambda':>8} | {'delta sim':>12} | {'delta exact':>12} | {'difference':>10} | {'max |H|':>8}")
print("-" * 70)
for (a, lmd), r in sorted(res.items()):
    exact = analytic.deflection_exact_deg(M, a, lmd, b, r0, r.r[-1])
    print(f"{a:>6} {lmd:>8.0e} | {r.deflection_deg:>12.6f} | {exact:>12.6f} | {r.deflection_deg - exact:>+10.1e} | "
          f"{np.max(np.abs(r.H)):>8.1e}")

     a   Lambda |    delta sim |  delta exact | difference |  max |H|
----------------------------------------------------------------------
 -0.99   -5e-06 |   245.785473 |   245.785472 |   +5.3e-07 |  5.0e-11
 -0.99    0e+00 |   246.062496 |   246.062495 |   +5.4e-07 |  5.0e-11
 -0.99    5e-06 |   246.202186 |   246.202186 |   +5.5e-07 |  5.0e-11
   0.0   -5e-06 |    64.906783 |    64.906783 |   +2.8e-09 |  2.0e-11
   0.0    0e+00 |    64.605781 |    64.605781 |   +2.8e-09 |  2.0e-11
   0.0    5e-06 |    64.229290 |    64.229290 |   +2.8e-09 |  2.0e-11
  0.99   -5e-06 |    45.644971 |    45.644971 |   +2.8e-09 |  2.2e-11
  0.99    0e+00 |    45.508988 |    45.508988 |   +2.8e-09 |  2.2e-11
  0.99    5e-06 |    45.372598 |    45.372598 |   +2.8e-09 |  2.2e-11
